# Donchian Channel Breakout (20-day) on SPY
## Strategy Brief
The Donchian Channel Breakout strategy is a trend-following strategy that uses the highest high and lowest low over a specific period, in this case, 20 days, to generate trading signals. When the price breaks above the 20-day high, it signals a buy, and when it breaks below the 20-day low, it signals a sell. The strategy aims to capture significant trends in the market, assuming that a breakout from a range will lead to a new trend. Historically, this strategy can be profitable in trending markets but may underperform in sideways markets.
## References
- https://jane.app/sign-in

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters of our trading strategy. The primary parameter for the Donchian Channel Breakout is the lookback period, which is set to 20 days. Additionally, we specify the ticker symbol for SPY and the start date for our historical data analysis.

In [ ]:
LOOKBACK_PERIOD = 20
TICKER = 'SPY'
START_DATE = '2010-01-01'

## PHASE 2 - Data Exploration
We will download historical price data for SPY from Yahoo Finance starting from January 1, 2010, to the present day. We will compute the Donchian Channel indicators, which include the 20-day high and 20-day low, and plot these indicators overlaid on the price data.

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE)

# Calculate Donchian Channel
highs = data['High'].rolling(window=LOOKBACK_PERIOD).max()
lows = data['Low'].rolling(window=LOOKBACK_PERIOD).min()

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='Close Price')
plt.plot(highs, label='20-Day High', linestyle='--')
plt.plot(lows, label='20-Day Low', linestyle='--')
plt.title('SPY Price with Donchian Channel')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
We will create a signal series based on the Donchian Channel breakout logic. A buy signal is generated when the closing price exceeds the 20-day high, and a sell signal is generated when the closing price falls below the 20-day low. We will then define the entry and exit logic and compute the positions series.

In [ ]:
buy_signal = data['Close'] > highs
sell_signal = data['Close'] < lows

# Define positions: 1 for long, -1 for short, 0 for no position
positions = pd.Series(index=data.index, data=0)
positions[buy_signal] = 1
positions[sell_signal] = -1

## PHASE 4 - Coding & Backtesting
We will implement the backtesting logic by shifting the positions by one day to avoid look-ahead bias. We will then calculate the daily returns and plot the equity curve of the strategy.

In [ ]:
positions = positions.shift(1).fillna(0)
daily_returns = data['Close'].pct_change().fillna(0)
strategy_returns = positions * daily_returns

# Calculate equity curve
equity_curve = (1 + strategy_returns).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(equity_curve, label='Strategy Equity Curve')
plt.title('Equity Curve of Donchian Channel Breakout Strategy')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
In this phase, we will evaluate the performance of the strategy using key performance metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and maximum drawdown. We will also compare these metrics against a buy-and-hold strategy for SPY.

In [ ]:
import numpy as np

# Calculate performance metrics
days = (data.index[-1] - data.index[0]).days
cagr = (equity_curve[-1])**(365.0/days) - 1

risk_free_rate = 0.01
sharpe_ratio = (strategy_returns.mean() - risk_free_rate/252) / strategy_returns.std() * np.sqrt(252)

negative_returns = strategy_returns[strategy_returns < 0]
sortino_ratio = (strategy_returns.mean() - risk_free_rate/252) / negative_returns.std() * np.sqrt(252)

max_drawdown = (equity_curve.cummax() - equity_curve).max()
calmar_ratio = cagr / max_drawdown

# Buy and hold comparison
equity_bh = (1 + daily_returns).cumprod()
cagr_bh = (equity_bh[-1])**(365.0/days) - 1

print(f"CAGR: {cagr:.2%}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Sortino Ratio: {sortino_ratio:.2f}")
print(f"Calmar Ratio: {calmar_ratio:.2f}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Buy and Hold CAGR: {cagr_bh:.2%}")

## PHASE 6 - Deploy & Monitor
Finally, we will create a function that downloads the last 60 days of SPY data, computes the Donchian Channel breakout signal for today, and prints the current position (long, short, or no position).

In [ ]:
def get_current_signal():
    recent_data = yf.download(TICKER, period='60d')
    recent_highs = recent_data['High'].rolling(window=LOOKBACK_PERIOD).max()
    recent_lows = recent_data['Low'].rolling(window=LOOKBACK_PERIOD).min()
    
    if recent_data['Close'][-1] > recent_highs[-1]:
        print("Current Position: Long")
    elif recent_data['Close'][-1] < recent_lows[-1]:
        print("Current Position: Short")
    else:
        print("Current Position: No Position")

get_current_signal()